# Our solution

The working notebook for our submission. The code lives in `starter/`, and this notebook uses it.

**Keep this notebook free of match data.** Print shapes and counts, not rows of tracking or tags, and clear all outputs before committing. The repository has to be public, and the data is not ours to publish.

## Load the data

The first run parses the tracking in about 5 seconds and saves it to `cache/`. Later runs load from there.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np

import games
import load_data

tracking = load_data.load_tracking()
tags = load_data.load_tags()

players = tracking['players']              # (frames, tracks, 2), metres
speeds = tracking['speeds']                # (frames, tracks), m/s
ball_raw = tracking['ball_raw']            # (frames, 2), metres
team_of_track = tracking['team_of_track']  # (tracks,), 0 = home, 1 = away

for name in load_data.ARRAYS:
    print('%-14s %s' % (name, tracking[name].shape))
print('%-14s %d rows, columns: %s' % ('tags', len(tags), ', '.join(tags.columns)))

players        (199319, 46, 2)
speeds         (199319, 46)
ball_raw       (199319, 2)
team_of_track  (46,)
tags           638 rows, columns: code, team, half, type, side, direction, max_players_box, start_s, end_s, start_frame, end_frame


## Using the arrays

Every array is indexed by video frame, so `players[f]` is row `f` of the submission. Empty slots and a missing ball are `NaN`.

Check `players`, not `speeds`, to see whether a track is present: about 9.8% of tracked positions have no speed.

In [2]:
present = ~np.isnan(players[:, :, 0])      # (frames, tracks), True where the slot is filled
ball_seen = ~np.isnan(ball_raw[:, 0])

graded = np.zeros(len(players), dtype=bool)
for half, first, last in games.half_video_blocks(games.GAME):
    graded[first:last + 1] = True

print('graded frames:              %d' % graded.sum())
print('ball seen on graded frames: %.1f%%' % (100 * ball_seen[graded].mean()))
print('tracks per graded frame:    %.1f' % present[graded].sum(axis=1).mean())

# Average team speed per frame, skipping NaN. np.nanmean warns on a frame with no speeds at all.
team_speed = np.nansum(speeds, axis=1) / np.maximum((~np.isnan(speeds)).sum(axis=1), 1)
print('median team speed, graded:  %.2f m/s' % np.median(team_speed[graded]))

graded frames:              158857
ball seen on graded frames: 59.3%
tracks per graded frame:    20.5
median team speed, graded:  1.51 m/s


## Using the tag table

All text is lower case, and `end_frame` is the first frame after the tag, so a tag's frames are `start_frame:end_frame`.

In [3]:
set_pieces = tags[tags['code'] == 'set pieces']
print(set_pieces['type'].value_counts().to_string())
print()
print(tags['team'].value_counts().to_string())

type
throw-in                 26
corner kick              12
goal kick                10
free kick                 5
kick off (after goal)     4
kick off (start)          2

team
barcelona       314
atletic club    288
n/a              36
